# Claims Processing Efficiency Analysis

**Objective:** Analyze denial patterns, processing cycle times, clean claim rates, and reimbursement ratios across 8,500+ claims to identify operational bottlenecks and revenue recovery opportunities.

**Data Source:** Multi-payer claims dataset (Commercial, Medicare, Medicaid)  
**Analyst:** Meagan Parsons  
**Date:** June 2026

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_palette('Set2')

df = pd.read_csv('../data/processed/claims_efficiency_clean.csv', parse_dates=['service_date', 'submitted_date', 'decision_date'])

print(f'Total claims: {len(df):,}')
print(f'Date range: {df["service_date"].min().date()} to {df["service_date"].max().date()}')
print(f'Unique payers: {df["payer"].nunique()} — {df["payer"].unique().tolist()}')
print(f'Unique specialties: {df["provider_specialty"].nunique()}')
print(f'\nClaim status breakdown:')
print(df['claim_status'].value_counts().to_string())
df.head(3)

## 1. Denial Rate Analysis by Payer and Specialty

Denial rates are a primary indicator of claims processing health. Stratifying by payer and specialty reveals which combinations drive the most rework and revenue delay.

In [ ]:
# Denial rate by payer
payer_denial = df.groupby('payer').agg(
    total_claims=('claim_id', 'count'),
    denied=('is_denied', 'sum'),
    reworked=('is_reworked', 'sum'),
    approved=('is_approved', 'sum'),
    avg_billed=('billed_amount', 'mean'),
    avg_paid=('paid_amount', 'mean')
).reset_index()
payer_denial['denial_rate'] = payer_denial['denied'] / payer_denial['total_claims']
payer_denial['rework_rate'] = payer_denial['reworked'] / payer_denial['total_claims']
payer_denial['reimbursement_ratio'] = payer_denial['avg_paid'] / payer_denial['avg_billed']

# Denial rate by specialty
spec_denial = df.groupby('provider_specialty').agg(
    total_claims=('claim_id', 'count'),
    denied=('is_denied', 'sum'),
    reworked=('is_reworked', 'sum')
).reset_index()
spec_denial['denial_rate'] = spec_denial['denied'] / spec_denial['total_claims']
spec_denial = spec_denial.sort_values('denial_rate', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Payer denial + rework stacked bar
x_pos = range(len(payer_denial))
axes[0].bar(x_pos, payer_denial['denial_rate'] * 100, label='Denied', color='#e74c3c', edgecolor='white', linewidth=0.3)
axes[0].bar(x_pos, payer_denial['rework_rate'] * 100, bottom=payer_denial['denial_rate'] * 100,
            label='Reworked', color='#f39c12', edgecolor='white', linewidth=0.3)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(payer_denial['payer'], rotation=30, ha='right')
axes[0].set_ylabel('Rate (%)', fontsize=11)
axes[0].set_title('Denial & Rework Rate by Payer', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)

# Specialty denial rate
axes[1].barh(spec_denial['provider_specialty'], spec_denial['denial_rate'] * 100,
             color='#3498db', edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('Denial Rate (%)', fontsize=11)
axes[1].set_title('Denial Rate by Provider Specialty', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('\nPayer Denial Summary:')
print(payer_denial[['payer', 'total_claims', 'denial_rate', 'rework_rate', 'reimbursement_ratio']].to_string(index=False))

## 2. Processing Cycle Time Analysis

Processing days (`decision_date - submitted_date`) directly impact cash flow. Segmenting by claim status and payer reveals where delays concentrate.

In [ ]:
# Cycle time statistics
cycle_by_status = df.groupby('claim_status')['processing_days'].agg(['mean', 'median', 'std', 'count']).reset_index()
cycle_by_status.columns = ['Claim Status', 'Mean Days', 'Median Days', 'Std Dev', 'Count']
print('Cycle Time by Claim Status:')
print(cycle_by_status.to_string(index=False))

# Kruskal-Wallis test: do processing days differ by claim status?
groups_status = [g['processing_days'].values for _, g in df.groupby('claim_status')]
h_stat, h_p = stats.kruskal(*groups_status)
print(f'\nKruskal-Wallis test (processing days ~ status): H={h_stat:.2f}, p={h_p:.4e}')

# Claims aging buckets
bins = [0, 15, 30, 45, 60, 90, 120, 365]
labels = ['0-15', '16-30', '31-45', '46-60', '61-90', '91-120', '120+']
df['aging_bucket'] = pd.cut(df['processing_days'], bins=bins, labels=labels, right=True)

aging = df.groupby(['aging_bucket', 'claim_status'], observed=True).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot of processing days by payer
payer_order = df.groupby('payer')['processing_days'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='payer', y='processing_days', order=payer_order, ax=axes[0],
            flierprops={'marker': '.', 'markersize': 2, 'alpha': 0.3})
axes[0].set_xlabel('Payer', fontsize=11)
axes[0].set_ylabel('Processing Days', fontsize=11)
axes[0].set_title('Processing Cycle Time Distribution by Payer', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Stacked aging bucket chart
aging.plot(kind='bar', stacked=True, ax=axes[1], edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('Aging Bucket (Days)', fontsize=11)
axes[1].set_ylabel('Claim Count', fontsize=11)
axes[1].set_title('Claims Aging Distribution by Status', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=8, title='Status')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 3. Clean Claim Rate and Reimbursement Ratios

The clean claim rate (first-pass approval with touch count = 1) is an industry benchmark metric. Target is typically 90%+.

In [ ]:
# Clean claim = approved on first touch
df['clean_claim'] = (df['is_approved'] == 1) & (df['touch_count'] == 1)

clean_by_payer = df.groupby('payer').agg(
    total=('claim_id', 'count'),
    clean=('clean_claim', 'sum'),
    total_billed=('billed_amount', 'sum'),
    total_paid=('paid_amount', 'sum')
).reset_index()
clean_by_payer['clean_rate'] = clean_by_payer['clean'] / clean_by_payer['total']
clean_by_payer['reimbursement_ratio'] = clean_by_payer['total_paid'] / clean_by_payer['total_billed']

# Monthly trend of clean claim rate
df['service_month'] = df['service_date'].dt.to_period('M')
monthly = df.groupby('service_month').agg(
    total=('claim_id', 'count'),
    clean=('clean_claim', 'sum'),
    denied=('is_denied', 'sum'),
    avg_processing=('processing_days', 'mean'),
    total_billed=('billed_amount', 'sum'),
    total_paid=('paid_amount', 'sum')
).reset_index()
monthly['clean_rate'] = monthly['clean'] / monthly['total']
monthly['denial_rate'] = monthly['denied'] / monthly['total']
monthly['reimb_ratio'] = monthly['total_paid'] / monthly['total_billed']
monthly['month_str'] = monthly['service_month'].astype(str)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Clean claim rate by payer
axes[0, 0].bar(clean_by_payer['payer'], clean_by_payer['clean_rate'] * 100,
               color='#2ecc71', edgecolor='white', linewidth=0.3)
axes[0, 0].axhline(y=90, color='#e74c3c', linestyle='--', label='90% Target')
axes[0, 0].set_ylabel('Clean Claim Rate (%)', fontsize=11)
axes[0, 0].set_title('Clean Claim Rate by Payer', fontsize=13, fontweight='bold')
axes[0, 0].legend(fontsize=9)

# Monthly clean claim rate trend
axes[0, 1].plot(monthly['month_str'], monthly['clean_rate'] * 100, marker='o', color='#2ecc71', linewidth=2)
axes[0, 1].axhline(y=90, color='#e74c3c', linestyle='--', alpha=0.6)
axes[0, 1].set_ylabel('Clean Claim Rate (%)', fontsize=11)
axes[0, 1].set_title('Monthly Clean Claim Rate Trend', fontsize=13, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)

# Monthly reimbursement ratio
axes[1, 0].plot(monthly['month_str'], monthly['reimb_ratio'] * 100, marker='s', color='#3498db', linewidth=2)
axes[1, 0].set_ylabel('Reimbursement Ratio (%)', fontsize=11)
axes[1, 0].set_title('Monthly Reimbursement Ratio Trend', fontsize=13, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)

# Monthly avg processing days
axes[1, 1].plot(monthly['month_str'], monthly['avg_processing'], marker='^', color='#e67e22', linewidth=2)
axes[1, 1].set_ylabel('Avg Processing Days', fontsize=11)
axes[1, 1].set_title('Monthly Average Processing Time', fontsize=13, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

overall_clean = df['clean_claim'].mean()
print(f'Overall clean claim rate: {overall_clean:.1%}')
print(f'Overall reimbursement ratio: {df["paid_amount"].sum() / df["billed_amount"].sum():.1%}')

## 4. Pareto Analysis of Denial Reasons

Identifying the vital few denial reasons that drive the majority of denials (80/20 rule) focuses remediation efforts where they will have the greatest impact.

In [ ]:
# Pareto analysis on denial reasons (exclude approved/not applicable)
denied_df = df[df['denial_reason'] != 'Not Applicable'].copy()

denial_counts = denied_df['denial_reason'].value_counts().reset_index()
denial_counts.columns = ['denial_reason', 'count']
denial_counts['cumulative_pct'] = denial_counts['count'].cumsum() / denial_counts['count'].sum() * 100
denial_counts['individual_pct'] = denial_counts['count'] / denial_counts['count'].sum() * 100

# Revenue impact of each denial reason
denial_revenue = denied_df.groupby('denial_reason')['billed_amount'].sum().reset_index()
denial_revenue.columns = ['denial_reason', 'total_billed_at_risk']
denial_counts = denial_counts.merge(denial_revenue, on='denial_reason')

fig, ax1 = plt.subplots(figsize=(14, 6))

bars = ax1.bar(range(len(denial_counts)), denial_counts['count'],
               color='#e74c3c', edgecolor='white', linewidth=0.3, alpha=0.85)
ax1.set_xticks(range(len(denial_counts)))
ax1.set_xticklabels(denial_counts['denial_reason'], rotation=35, ha='right', fontsize=9)
ax1.set_ylabel('Denial Count', fontsize=11, color='#e74c3c')

ax2 = ax1.twinx()
ax2.plot(range(len(denial_counts)), denial_counts['cumulative_pct'],
         marker='o', color='#f1c40f', linewidth=2.5, markersize=6)
ax2.axhline(y=80, color='#2ecc71', linestyle='--', alpha=0.7, label='80% Threshold')
ax2.set_ylabel('Cumulative %', fontsize=11, color='#f1c40f')
ax2.set_ylim(0, 105)
ax2.legend(loc='center right', fontsize=9)

ax1.set_title('Pareto Analysis of Denial Reasons', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Identify the 80% threshold
threshold_idx = (denial_counts['cumulative_pct'] >= 80).idxmax()
vital_few = denial_counts.iloc[:threshold_idx + 1]
print(f'Vital few denial reasons (driving 80% of denials): {len(vital_few)}')
print(f'Total billed amount at risk from vital few: ${vital_few["total_billed_at_risk"].sum():,.0f}')
print()
print(denial_counts[['denial_reason', 'count', 'individual_pct', 'cumulative_pct', 'total_billed_at_risk']].to_string(index=False))

## 5. Payer-Specialty Cross-Tabulation and Statistical Testing

Testing whether denial rates are independent of the payer-specialty combination using chi-square analysis.

In [ ]:
# Pivot table: denial rate by payer x specialty
pivot = pd.pivot_table(df, values='is_denied', index='provider_specialty', columns='payer',
                       aggfunc='mean').round(3)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot * 100, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Denial Rate (%)'})
ax.set_title('Denial Rate (%) by Payer and Specialty', fontsize=13, fontweight='bold')
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

# Chi-square test: denial status vs. payer
ct_payer = pd.crosstab(df['payer'], df['is_denied'])
chi2, p_val, dof, exp = stats.chi2_contingency(ct_payer)
print(f'Chi-square (denial ~ payer): chi2={chi2:.2f}, p={p_val:.4e}, dof={dof}')

# Chi-square test: denial status vs. specialty
ct_spec = pd.crosstab(df['provider_specialty'], df['is_denied'])
chi2_s, p_s, dof_s, _ = stats.chi2_contingency(ct_spec)
print(f'Chi-square (denial ~ specialty): chi2={chi2_s:.2f}, p={p_s:.4e}, dof={dof_s}')

## Key Findings

1. **Denial rate varies significantly by payer** (chi-square test), with certain payers consistently generating higher denial and rework volumes. Targeted payer-specific submission rules could reduce initial denials.

2. **Reworked claims have substantially longer cycle times** than first-pass approvals, directly impacting days-in-AR and cash conversion. Each rework touch adds measurable delay.

3. **Clean claim rate** falls below the 90% industry target for most payer segments. Improving front-end eligibility verification and coding accuracy on the vital-few denial reasons would move the needle most efficiently.

4. **Pareto analysis** confirms that a small number of denial reasons (missing documentation, coding errors, eligibility issues) account for the majority of denials — a focused training initiative on these categories should yield outsized returns.

5. **Reimbursement ratios** show month-over-month variation that warrants monitoring; dips may correlate with payer contract changes or coding pattern shifts.